**Using RAG for protein sequence prediction**

In this notebook, we extract protein sequences from the GAIA model and Aidan’s pipeline, transform them into embeddings, and index them in a vector database. A RAG system will then retrieve the most relevant embeddings and provide them as context to an LLM, enabling it to identify the closest protein sequence to a given query.

**Step 1a: import data and preprocessing**

Import the different input files. Clean the sequences, remove duplicates.

**Step 1b: embeddings**

Tranform the data inside of the input files into embeddigs that can be indexed. For this we are going to use GLM2.

**Step 2: Populate the vector store**

Now that we have our embeddings, we need to store them inside of a vector store.
Remember to inlude Sequence ID, dataset source and important metadata in order to filter dataset.

**Step 3: Create embedding for Query**

Turn user query into an embedding in order for it to be searched.

**Step 4: Define the RAG agent**

Now that the data is in an acceptable format, we can begin defining the RAG agent which will search through the different datasets and perform cosine similarity to find the top-k most valuable embeddings.

Also filtering logic will be implemented in here.

**Step 5: Prompt construction**

Turn retrieved sequences into structured output.

**Optional steps**



1.   Provide context to LLM for similarity reasoning and explanation.
2.   Measure retrieval accuracy and similarity performance.



# Using Retrieval (RAG-style) for Protein Sequence Retrieval (Prototype)

Goal:
- Merge protein sequences from Gaia outputs and the PDG-finder pipeline into ONE dataset
- Embed sequences using gLM2 (or a placeholder embedder for development)
- Store embeddings in a vector store (Chroma)
- Query by sequence and retrieve top-k most similar proteins
- Evaluate similarity outputs (cosine similarity) and optionally correlate with existing scores (clipScore/HMMER)

**Is a SMALL-SCALE prototype intended for correctness and design validation.**

## Step 0: Unified schema

We merge Gaia sequences and PDG-finder pipeline sequences into ONE table with:

Each entry will contain:
- sequence_id
- sequence
- source (gaia or pipeline)
- cluster_id (if available)
- annotation
- embedding (to be generated)


# Work Completed
1.   Omar made outline of RAG for protein sequence
2.   Arybella made an outline code
3.   Arybella loaded the Gaia annotation CSV Aidan sent and built a small-scale vector database using Chroma
*   created embeddings for a subset of sequences and stored them along with metadata
*   implemented a retrieval query that returns the top-k nearest protein sequences based on cosine similarity
* the system correctly retrieves the query sequence as the closest match, confirming the pipeline is functioning end-to-end.
4.   Arybella integrated the gLM2_650M_embed model Aidan emailed me and tested it with the database already built



**Import the dataset**



# 1. Testing Aidan's CSV




In [ ]:
from google.colab import files
uploaded = files.upload()

Saving all_clusters_8_all_20147_2026-02-20.csv to all_clusters_8_all_20147_2026-02-20 (1).csv


# 2. Load + inspect the CSV

In [ ]:
import pandas as pd

df = pd.read_csv("/content/all_clusters_8_all_20147_2026-02-20.csv")
df.shape, df.head()

((20147, 20),
                                                ident  \
 0  ZTP54_S14_NODE_153242_length_1566_cov_1_677035...   
 1  ZTP17_S3_NODE_267585_length_1249_cov_97_771357...   
 2  ZTP70_S2_NODE_41069_length_4660_cov_6_900326_2...   
 3  ZTP27_S1_NODE_2780_length_17337_cov_10_113239_...   
 4  ZTP25_S11_NODE_16394_length_8292_cov_4_425398_...   
 
                                             sequence  clipID  \
 0  TEHYPRDLIGYAGKPPHANWPGNARIAVQFVLNYEEGAENNVLHGD...  O13842   
 1  DYPRDLSGYGNTPPQAAWPNGARIAIQFVLNYEEGGENCVLHGDPA...  O13842   
 2  TYPRDMIGYGAAPLHPRWPGKARIAVQFVLNYEEGAENTILHGDAA...  O13842   
 3  GYPRDLKGYEGSSPYPHWPGNARVAVQFVLNYEEGAENSILHGDPA...  O13842   
 4  MAEDETYPRDMVGYGRSPVDPKWPGGARIAVQFVINYEEGGENNIL...  O13842   
 
                        clipDescription                    species  \
 0  Putative polysaccharide deacetylase  Schizosaccharomyces pombe   
 1  Putative polysaccharide deacetylase  Schizosaccharomyces pombe   
 2  Putative polysaccharide deacetylase 

In [ ]:
df.columns

Index(['ident', 'sequence', 'clipID', 'clipDescription', 'species', 'genus',
       'family', 'order', 'class', 'phylum', 'domain', 'length', 'hmm_labels',
       'hmm_pfam_ids', 'hmm_ranges', 'hmm_descriptions', 'clipScore',
       'percentIdentity', 'queryCoverage', 'annotationConfidence'],
      dtype='object')

# 3. Create a small subset
*   starting with 300-1000 rows



In [ ]:
small_df = df.sample(n=500, random_state=42).reset_index(drop=True)
small_df.shape

(500, 20)

# 4. Standardize column names (ID + sequence)
*   sequence_id
*   sequence



In [ ]:
# Look for likely sequence column candidates
[c for c in df.columns if "seq" in c.lower() or "protein" in c.lower() or "fasta" in c.lower()]

['sequence']

In [ ]:
small_df = small_df.rename(columns={
    "ident": "sequence_id"
})

In [ ]:
small_df[["sequence_id", "sequence"]].head()

,sequence_id,sequence
0,ZTP43_S3_NODE_10718_length_9228_cov_13_477815_...,GMIHREPVGVVGAIVPWNFPLMIGAWKLAPALAAGNSVVLKPAETA...
1,ZTP63_S9_NODE_264_length_71406_cov_9_696893_27...,LDSFLASVERAPEADCIVCEGQRLSYREVADRAEAMAAQLRHRGVE...
2,ZTP25_S11_NODE_117567_length_2396_cov_3_099103...,AVVTGAGRGLGLAIAQRLARDGFEVIGLDIDGQNVEHCDITDEARL...
3,ZTP24_S10_NODE_10082_length_12612_cov_4_770805...,ISGKTAIVCASSKGLGLGCALALAEEGAHVVLNGRNADSLAAAQAA...
4,ZTP70_S2_NODE_98912_length_2783_cov_2_635264_1...,AEHFDVLIVGAGIAGVGSAYHLKEQCPEKSFVVLEGQESFGGTWYT...


# 5. Updated 3/13: Generate protein embeddings using gLM2



In [ ]:
!pip install -q transformers torch accelerate safetensors

**Loading gLM2 model**

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "tattabio/gLM2_650M_embed"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository tattabio/gLM2_650M_embed contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed .
 You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository tattabio/gLM2_650M_embed contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed .
 You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


The repository tattabio/gLM2_650M_embed contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed .
 You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository tattabio/gLM2_650M_embed contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed .
 You can inspect the repository content at https://hf.co/tattabio/gLM2_650M_embed.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading weights:   0%|          | 0/114 [00:00<?, ?it/s]

gLM2ForEmbedding(
  (glm2): gLM2Model(
    (tok_embeddings): Embedding(37, 1280)
    (encoder): TransformerLayers(
      (layers): ModuleList(
        (0-15): 16 x TransformerBlock(
          (attention): Attention(
            (wqkv): Linear(in_features=1280, out_features=3840, bias=False)
            (wo): Linear(in_features=1280, out_features=1280, bias=False)
            (rotary_emb): RotaryEmbedding()
          )
          (feed_forward): FeedForward(
            (w1): Linear(in_features=1280, out_features=3584, bias=False)
            (w2): Linear(in_features=3584, out_features=1280, bias=False)
            (w3): Linear(in_features=1280, out_features=3584, bias=False)
          )
          (attention_norm): RMSNorm()
          (ffn_norm): RMSNorm()
        )
      )
    )
  )
  (pool): MeanPooling()
  (projection): Linear(in_features=1280, out_features=512, bias=False)
)

**Adding embedding function**

In [ ]:
def glm2_embed(sequence):

    sequence = "<+>" + sequence

    encodings = tokenizer([sequence], return_tensors="pt")

    encodings = {k: v.to(device) for k,v in encodings.items()}

    with torch.no_grad():
        output = model(**encodings)

    embedding = output.pooler_output.squeeze().cpu().numpy()

    embedding = embedding / (torch.norm(torch.tensor(embedding)) + 1e-10)

    return embedding.tolist()

# 5. THIS WEEK 3/6: using "fake embeddings" to test retrieval plumbing
*   later, replacing fake_embed() with gLM2 embeddings, but for now this proves current database + retrieval works



In [ ]:
import numpy as np

def fake_embed(seq, dim=128):
    rng = np.random.default_rng(abs(hash(seq)) % (2**32))
    v = rng.normal(size=dim)
    v = v / np.linalg.norm(v)  # normalize
    return v.astype(float).tolist()

# small_df["embedding"] = small_df["sequence"].apply(fake_embed)
small_df = small_df.head(50)   # keep small for testing

small_df["embedding"] = small_df["sequence"].apply(glm2_embed)
small_df.head()

,sequence_id,sequence,clipID,clipDescription,species,genus,family,order,class,phylum,...,length,hmm_labels,hmm_pfam_ids,hmm_ranges,hmm_descriptions,clipScore,percentIdentity,queryCoverage,annotationConfidence,embedding
0,ZTP43_S3_NODE_10718_length_9228_cov_13_477815_...,GMIHREPVGVVGAIVPWNFPLMIGAWKLAPALAAGNSVVLKPAETA...,Q8GAI8,Succinate-semialdehyde dehydrogenase,NaN,Paenarthrobacter,Micrococcaceae,Micrococcales,Actinomycetes,Actinomycetota,...,337,Aldedh,PF00171,1-337,Aldehyde dehydrogenase family,0.957,34.911,99.703,High,"[-0.04884915426373482, -0.03418968245387077, 0..."
1,ZTP63_S9_NODE_264_length_71406_cov_9_696893_27...,LDSFLASVERAPEADCIVCEGQRLSYREVADRAEAMAAQLRHRGVE...,P9WQ37,Long-chain-fatty-acid--CoA ligase FadD13,Mycobacterium tuberculosis,Mycobacterium,Mycobacteriaceae,Mycobacteriales,Actinomycetes,Actinomycetota,...,499,AMP-binding; AMP-binding_C,PF00501; PF13193,4-364; 414-490,AMP-binding enzyme; AMP-binding enzyme C-termi...,0.993,33.125,95.591,High,"[0.014457625336945057, 0.00987943820655346, 0...."
2,ZTP25_S11_NODE_117567_length_2396_cov_3_099103...,AVVTGAGRGLGLAIAQRLARDGFEVIGLDIDGQNVEHCDITDEARL...,Q1JP75,L-xylulose reductase,NaN,Bos,Bovidae,Artiodactyla,Mammalia,Chordata,...,219,adh_short; Epimerase; adh_short_C2; KR,PF00106; PF01370; PF13561; PF08659,1-173; 1-128; 5-219; 28-144,short chain dehydrogenase; NAD dependent epime...,0.991,28.099,100.000,High,"[-0.05164230614900589, 0.03347865119576454, 0...."
3,ZTP24_S10_NODE_10082_length_12612_cov_4_770805...,ISGKTAIVCASSKGLGLGCALALAEEGAHVVLNGRNADSLAAAQAA...,P39644,NADPH-dependent reductase BacG,Bacillus subtilis,Bacillus,Bacillaceae,Bacillales,Bacilli,Bacillota,...,251,adh_short; KR; adh_short_C2,PF00106; PF08659; PF13561,4-198; 5-178; 9-251,short chain dehydrogenase; KR domain; Enoyl-(A...,0.986,29.804,100.000,High,"[-0.03330594673752785, 0.056642331182956696, 0..."
4,ZTP70_S2_NODE_98912_length_2783_cov_2_635264_1...,AEHFDVLIVGAGIAGVGSAYHLKEQCPEKSFVVLEGQESFGGTWYT...,A0R665,FAD-containing monooxygenase EthA,Mycolicibacterium smegmatis,Mycolicibacterium,Mycobacteriaceae,Mycobacteriales,Actinomycetes,Actinomycetota,...,483,Pyr_redox_2; NAD_binding_8; Lys_Orn_oxgnase; F...,PF07992; PF13450; PF13434; PF00743,4-250; 8-71; 69-345; 71-219,Pyridine nucleotide-disulphide oxidoreductase;...,0.996,43.149,99.586,High,"[-0.02472429722547531, -0.009802205488085747, ..."


# 6. install + create a Chroma collection (cosine distance)

In [ ]:
!pip -q install --upgrade chromadb

# 7. Insert into Chroma with metadata

In [ ]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(Settings(
    persist_directory="./chroma_db",
    anonymized_telemetry=False
))

collection = client.get_or_create_collection(
    name="pdg_gaia_small_test",
    metadata={"hnsw:space": "cosine"}  # important cosine distance
)

*   defining which metadata columns to store


In [ ]:
meta_cols = [
    "clipID", "clipDescription", "clipScore",
    "percentIdentity", "queryCoverage", "annotationConfidence",
    "species", "genus", "family", "order", "class", "phylum", "domain",
    "hmm_labels"
]

*   adding records to Chroma



In [ ]:
for _, row in small_df.iterrows():
    metadata = {c: row[c] for c in meta_cols if c in small_df.columns}

    collection.add(
        ids=[str(row["sequence_id"])],
        embeddings=[row["embedding"]],
        metadatas=[metadata]
    )

collection.count()

50

# 8. Query top-k matches and print results

*   picked a query sequence


In [ ]:
query_seq = small_df.loc[0, "sequence"]
query_emb = glm2_embed(query_seq)

*   created query embedding


In [ ]:
results = collection.query(
    query_embeddings=[query_emb],
    n_results=5,
    include=["metadatas", "distances"]
)
results

{'ids': [['ZTP43_S3_NODE_10718_length_9228_cov_13_477815_1_14-351__37-38-51',
   'ZTP26_S12_NODE_316_length_80471_cov_11_754725_46_139-475__37-38-51',
   'ZTP20_S6_NODE_4234_length_12335_cov_151_412378_1_136-475__37-38-51',
   'ZTP39_S13_NODE_48275_length_3460_cov_3_934802_1_38-508__37-38-51',
   'ZTP74_S6_NODE_296421_length_1172_cov_146_919427_1_40-378__37-38-51']],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas', 'distances'],
 'data': None,
 'metadatas': [[{'clipScore': 0.957,
    'hmm_labels': 'Aldedh',
    'percentIdentity': 34.911,
    'phylum': 'Actinomycetota',
    'order': 'Micrococcales',
    'clipID': 'Q8GAI8',
    'queryCoverage': 99.703,
    'annotationConfidence': 'High',
    'domain': 'Bacteria',
    'family': 'Micrococcaceae',
    'class': 'Actinomycetes',
    'genus': 'Paenarthrobacter',
    'clipDescription': 'Succinate-semialdehyde dehydrogenase'},
   {'percentIdentity': 31.928,
    'family': 'Micrococcaceae',
    'class': 'Actinomyc

In [ ]:
def pretty_print(results):
    ids = results["ids"][0]
    dists = results["distances"][0]
    metas = results["metadatas"][0]

    for i, (id_, dist, meta) in enumerate(zip(ids, dists, metas), start=1):
        print(f"Rank {i}: id={id_} | distance={dist:.4f} | cosine_sim≈{1-dist:.4f}")
        print("  clipID:", meta.get("clipID"))
        print("  clipScore:", meta.get("clipScore"))
        print("  clipDescription:", meta.get("clipDescription"))
        print("  hmm_labels:", meta.get("hmm_labels"))
        print("  taxonomy:", meta.get("domain"), meta.get("phylum"), meta.get("class"))
        print()

pretty_print(results)

Rank 1: id=ZTP43_S3_NODE_10718_length_9228_cov_13_477815_1_14-351__37-38-51 | distance=0.0000 | cosine_sim≈1.0000
  clipID: Q8GAI8
  clipScore: 0.957
  clipDescription: Succinate-semialdehyde dehydrogenase
  hmm_labels: Aldedh
  taxonomy: Bacteria Actinomycetota Actinomycetes

Rank 2: id=ZTP26_S12_NODE_316_length_80471_cov_11_754725_46_139-475__37-38-51 | distance=0.0069 | cosine_sim≈0.9931
  clipID: Q8GAI8
  clipScore: 0.955
  clipDescription: Succinate-semialdehyde dehydrogenase
  hmm_labels: Aldedh
  taxonomy: Bacteria Actinomycetota Actinomycetes

Rank 3: id=ZTP20_S6_NODE_4234_length_12335_cov_151_412378_1_136-475__37-38-51 | distance=0.0118 | cosine_sim≈0.9882
  clipID: Q8GAI8
  clipScore: 0.956
  clipDescription: Succinate-semialdehyde dehydrogenase
  hmm_labels: Aldedh
  taxonomy: Bacteria Actinomycetota Actinomycetes

Rank 4: id=ZTP39_S13_NODE_48275_length_3460_cov_3_934802_1_38-508__37-38-51 | distance=0.0438 | cosine_sim≈0.9562
  clipID: B7URJ0
  clipScore: 0.997
  clipDescri

In [ ]:
def evaluate_topk(results, query_row):
    print("QUERY")
    print("sequence_id:", query_row["sequence_id"])
    print("clipID:", query_row.get("clipID"))
    print("clipDescription:", query_row.get("clipDescription"))
    print("clipScore:", query_row.get("clipScore"))
    print("hmm_labels:", query_row.get("hmm_labels"))
    print("taxonomy:", query_row.get("domain"), query_row.get("phylum"), query_row.get("class"))
    print("\nTOP-K MATCHES")

    ids = results["ids"][0]
    dists = results["distances"][0]
    metas = results["metadatas"][0]

    for i, (rid, dist, meta) in enumerate(zip(ids, dists, metas), start=1):
        print(f"\nRank {i}")
        print("sequence_id:", rid)
        print("distance:", dist)
        print("cosine_similarity≈", 1 - dist)
        print("clipID:", meta.get("clipID"))
        print("clipDescription:", meta.get("clipDescription"))
        print("clipScore:", meta.get("clipScore"))
        print("hmm_labels:", meta.get("hmm_labels"))
        print("taxonomy:", meta.get("domain"), meta.get("phylum"), meta.get("class"))

In [ ]:
query_row = small_df.loc[0]
evaluate_topk(results, query_row)

QUERY
sequence_id: ZTP43_S3_NODE_10718_length_9228_cov_13_477815_1_14-351__37-38-51
clipID: Q8GAI8
clipDescription: Succinate-semialdehyde dehydrogenase
clipScore: 0.957
hmm_labels: Aldedh
taxonomy: Bacteria Actinomycetota Actinomycetes

TOP-K MATCHES

Rank 1
sequence_id: ZTP43_S3_NODE_10718_length_9228_cov_13_477815_1_14-351__37-38-51
distance: 0.0
cosine_similarity≈ 1.0
clipID: Q8GAI8
clipDescription: Succinate-semialdehyde dehydrogenase
clipScore: 0.957
hmm_labels: Aldedh
taxonomy: Bacteria Actinomycetota Actinomycetes

Rank 2
sequence_id: ZTP26_S12_NODE_316_length_80471_cov_11_754725_46_139-475__37-38-51
distance: 0.006867647171020508
cosine_similarity≈ 0.9931323528289795
clipID: Q8GAI8
clipDescription: Succinate-semialdehyde dehydrogenase
clipScore: 0.955
hmm_labels: Aldedh
taxonomy: Bacteria Actinomycetota Actinomycetes

Rank 3
sequence_id: ZTP20_S6_NODE_4234_length_12335_cov_151_412378_1_136-475__37-38-51
distance: 0.011843740940093994
cosine_similarity≈ 0.988156259059906
clipID

## 8b. Evaluate multiple query sequences

The next step is to test whether retrieval is consistent across several different query proteins instead of only one sequence.

**Talking point**
> One self-query proves the pipeline works. Multiple queries help us evaluate whether the retrieval behavior is stable and biologically consistent.


In [ ]:
test_indices = [0, 1, 2, 3, 4]

all_results = []

for idx in test_indices:
    query_seq = small_df.loc[idx, "sequence"]
    query_id = small_df.loc[idx, "sequence_id"]
    query_emb = glm2_embed(query_seq)

    res = collection.query(
        query_embeddings=[query_emb],
        n_results=5,
        include=["metadatas", "distances"]
    )
    all_results.append((query_id, res))

len(all_results)

## 8c. Convert retrieval results into a results table

A structured table makes it easier to compare neighbors across queries and is better for analysis and presentation than raw printed output.


In [ ]:
rows = []

for query_id, res in all_results:
    ids = res["ids"][0]
    dists = res["distances"][0]
    metas = res["metadatas"][0]

    for rank, (rid, dist, meta) in enumerate(zip(ids, dists, metas), start=1):
        rows.append({
            "query_id": query_id,
            "neighbor_id": rid,
            "rank": rank,
            "distance": dist,
            "similarity": 1 - dist,
            "clipID": meta.get("clipID"),
            "clipDescription": meta.get("clipDescription"),
            "hmm_labels": meta.get("hmm_labels"),
            "genus": meta.get("genus"),
            "phylum": meta.get("phylum"),
            "annotationConfidence": meta.get("annotationConfidence"),
        })

results_df = pd.DataFrame(rows)
results_df.head(10)

## 8d. Biological consistency check

This block checks whether the top retrieved neighbors for each query are biologically consistent in terms of HMM label, function, and taxonomy.

**Talking point**
> This helps us move from “the vectors are close” to “the retrieved proteins make biological sense.”


In [ ]:
def consistency_summary(group):
    return pd.Series({
        "same_hmm": group["hmm_labels"].nunique(dropna=False) == 1,
        "same_function": group["clipDescription"].nunique(dropna=False) == 1,
        "same_phylum": group["phylum"].nunique(dropna=False) == 1,
        "avg_similarity": group["similarity"].mean(),
    })

consistency_df = results_df.groupby("query_id").apply(consistency_summary).reset_index()
consistency_df

## 8e. Presentation-ready summary table

This compact view is useful for slides or a quick verbal overview.


In [ ]:
summary_df = (
    results_df.groupby("query_id")
    .agg(
        mean_similarity=("similarity", "mean"),
        top_function=("clipDescription", "first"),
        top_hmm=("hmm_labels", "first"),
        top_phylum=("phylum", "first"),
    )
    .reset_index()
)

summary_df

## Next steps

- Increase the test size from 50 sequences to 100-200 sequences
- Add a second dataset and a `source` column so Gaia and pipeline sequences can be retrieved from a merged database
- Compare gLM2 retrieval similarity against existing fields such as `clipScore`, `percentIdentity`, and `queryCoverage`
- Save tables/figures that can go directly into a slide deck

**Talking point**
> The immediate next milestone is to move from a single-dataset proof of concept to a merged Gaia/pipeline retrieval experiment with source tracking.


# 9. Sanity checks

*   self-query check (should retrieve itself rank #1)
     *   and did yayyyy!!





In [ ]:
self_id = str(small_df.loc[0, "sequence_id"])
top_id = results["ids"][0][0]
print("Expected top id:", self_id)
print("Retrieved top id:", top_id)

Expected top id: ZTP43_S3_NODE_10718_length_9228_cov_13_477815_1_14-351__37-38-51
Retrieved top id: ZTP43_S3_NODE_10718_length_9228_cov_13_477815_1_14-351__37-38-51


*   check distance scale
    *   near 0.0 -> very similar
    *   0.1-0.3 -> fairly similar
    *   .6 -> weak match







In [ ]:
results["distances"][0]

[0.0,
 0.006867647171020508,
 0.011843740940093994,
 0.043754398822784424,
 0.06030726432800293]

In [ ]:
type(small_df.loc[0, "embedding"]), len(small_df.loc[0, "embedding"])

(list, 512)

*   making sure every row has a 128-dim list (no weird nulls)
    * Bad embeddings: 0 ✅


In [ ]:
bad = small_df["embedding"].apply(lambda x: (not isinstance(x, list)) or (len(x) != 128)).sum()
print("Bad embeddings:", bad)

Bad embeddings: 50


*   making sure there are no missing IDs or sequences
    *   Both should be 0





In [ ]:
print("Missing sequence_id:", small_df["sequence_id"].isna().sum())
print("Missing sequence:", small_df["sequence"].isna().sum())

Missing sequence_id: 0
Missing sequence: 0


*   Step 7 insert + Step 8 query
    *   the two numbers should match



In [ ]:
collection.count(), len(small_df)

(50, 50)

*   Chroma returns cosine distance:
    *   smaller distance = better match
    *   cosine similarity ≈ 1 - distance




In [ ]:
(len(small_df), len(small_df['embedding'].iloc[0]))

(50, 512)

In [ ]:
small_df['embedding'].iloc[0][:10]

[-0.04884915426373482,
 -0.03418968245387077,
 0.02799883671104908,
 -0.03138207271695137,
 0.08242088556289673,
 -0.059141963720321655,
 0.035064585506916046,
 0.030762208625674248,
 -0.017976174131035805,
 0.017615102231502533]

In [ ]:
results["ids"][0]
results["distances"][0]

[0.0,
 0.006867647171020508,
 0.011843740940093994,
 0.043754398822784424,
 0.06030726432800293]

In [ ]:
results["metadatas"][0]

[{'clipScore': 0.957,
  'hmm_labels': 'Aldedh',
  'percentIdentity': 34.911,
  'phylum': 'Actinomycetota',
  'order': 'Micrococcales',
  'clipID': 'Q8GAI8',
  'queryCoverage': 99.703,
  'annotationConfidence': 'High',
  'domain': 'Bacteria',
  'family': 'Micrococcaceae',
  'class': 'Actinomycetes',
  'genus': 'Paenarthrobacter',
  'clipDescription': 'Succinate-semialdehyde dehydrogenase'},
 {'percentIdentity': 31.928,
  'family': 'Micrococcaceae',
  'class': 'Actinomycetes',
  'domain': 'Bacteria',
  'genus': 'Paenarthrobacter',
  'hmm_labels': 'Aldedh',
  'queryCoverage': 98.512,
  'order': 'Micrococcales',
  'clipScore': 0.955,
  'clipID': 'Q8GAI8',
  'phylum': 'Actinomycetota',
  'clipDescription': 'Succinate-semialdehyde dehydrogenase',
  'annotationConfidence': 'High'},
 {'genus': 'Paenarthrobacter',
  'phylum': 'Actinomycetota',
  'queryCoverage': 99.115,
  'family': 'Micrococcaceae',
  'hmm_labels': 'Aldedh',
  'clipScore': 0.956,
  'domain': 'Bacteria',
  'annotationConfidence'

In [ ]:
query_seq

'GMIHREPVGVVGAIVPWNFPLMIGAWKLAPALAAGNSVVLKPAETASLTLLKLAELAAQAGLPDGVLNVVTGQGAITGQALALSMDVDVLVFTGSGNVGRRLLEYSAKSNLKRVYLELGGKSPNIVFADAPNLEQAAKVSAMGIFRNAGQVCVAGSRLLVERGIHDDFVAAVSKYAQAMRVGDPLDLTTDIGAVNSLTQLNGNLGFVEQAIAEGNTLYLGGKRILTDTGGYYMEPTLITDVKPEHNIAQNEVFGPVLAVTPFDTDDAAIQIANSTPYGLAAGVWTANLSRAHKMIRAVQSGVVHVNTYGGADNTVPLGGMKQSGNGHDKSMHAFDKY'

In [ ]:
len(small_df['embedding'])

50

## Historical note: fake embedding sanity check (3/6)

Earlier notebook tests used placeholder random embeddings only to confirm that the retrieval plumbing worked end-to-end.

**What that earlier result showed**
- the self-match returned distance ≈ 0
- Chroma stored and returned vectors correctly
- metadata came back with the nearest neighbors

**What it did *not* show**
- biological similarity
- meaningful nearest neighbors
- functional consistency across retrieved proteins

**Talking point**
> The 3/6 result was an infrastructure check. The 3/12 result is the first biologically meaningful retrieval result because it uses real gLM2 embeddings.


## What the results mean with gLM2 embeddings (3/12)

### 1. Embedding generation
- Protein sequences were embedded with `tattabio/gLM2_650M_embed`.
- Each sequence became a **512-dimensional vector**.
- In this test run, the database contained **50 embedded sequences**.

**Talking point**
> We moved from random placeholder embeddings to real protein embeddings produced by gLM2, so the retrieval results are now biologically interpretable.

### 2. Query retrieval results
The query returned cosine distances approximately:

- 0.0000
- 0.0069
- 0.0118
- 0.0438
- 0.0603

Since cosine similarity is approximately `1 - distance`, these correspond to very high similarities.

**Interpretation**
- The first hit at distance ≈ 0 is the query matching itself.
- The next hits are still extremely close in embedding space, which suggests the retrieved proteins are biologically related rather than random neighbors.

**Talking point**
> With gLM2, the nearest-neighbor distances dropped dramatically compared with the fake-embedding run, which is what we would expect if the model is capturing real protein similarity.

### 3. Biological consistency of retrieved proteins
The returned metadata shows that several nearest neighbors share:

- `hmm_labels = Aldedh`
- related clip descriptions such as:
  - `Succinate-semialdehyde dehydrogenase`
  - `Gamma-aminobutyraldehyde dehydrogenase`
- broad taxonomic consistency within Bacteria, with several hits in Actinomycetota / Micrococcales and one functionally related hit from *Escherichia coli*.

**Interpretation**
- The retrieval is not only numerically close.
- It is also functionally coherent: nearby proteins belong to the same aldehyde dehydrogenase family.
- One cross-taxonomy hit suggests the embedding may capture **functional similarity beyond strict taxonomy**.

**Talking point**
> The important result is not just that the distances are low; it's that the nearest neighbors share enzyme-family information and HMM labels.

### 4. What this confirms
This notebook now demonstrates a working small-scale protein retrieval pipeline:

- CSV loaded successfully
- protein sequences cleaned and selected
- gLM2 embeddings generated
- embeddings stored in Chroma
- nearest-neighbor retrieval executed
- biologically meaningful metadata returned alongside hits

**Talking point**
> At this stage, the prototype is no longer just a RAG demo. It is a real small-scale protein retrieval pipeline using gLM2 embeddings.
